# Chapter 28 — Time Series and Forecasting

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements-optional.txt  # Colab only; skip locally

zsh:1: command not found: pip


zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 28 (`code/ch28/gen_daily.py` in the repository).

In [2]:
import numpy as np, pandas as pd

rng = np.random.default_rng(11)
start = pd.Timestamp("2023-01-01")
days  = pd.date_range(start, periods=730, freq="D")

t     = np.arange(len(days))
trend = 1850 + 1.6 * t
dow   = np.array([1.00, 0.94, 0.97, 1.02, 1.18, 1.32, 0.86])
week  = dow[days.dayofweek]
year  = 1 + 0.16 * np.sin(2 * np.pi * (t - 80) / 365.25)
noise = rng.normal(1, 0.055, len(days))

rev = trend * week * year * noise
dec = (days.month == 12) & (days.day >= 10) & (days.day <= 24)
rev[dec] *= 1.45
closed = (days.month == 12) & (days.day == 25)
rev[closed] = 0

df = pd.DataFrame({"Date": days.strftime("%Y-%m-%d"),
                   "Revenue": rev.round(2)})
df.to_csv("daily_revenue.csv", index=False)
print(f"wrote daily_revenue.csv: {len(df):,} days, "
      f"{df.Date.min()} to {df.Date.max()}")

wrote daily_revenue.csv: 730 days, 2023-01-01 to 2024-12-30


## The chapter code

### Block 1  (`c1.py`)

In [3]:
import pandas as pd

s = pd.read_csv("daily_revenue.csv", parse_dates=["Date"])
s = s.set_index("Date")["Revenue"]

print(s.head(3))
print()
print(f"span:    {s.index.min().date()} to {s.index.max().date()}")
print(f"days:    {len(s):,}")
print(f"missing: {s.isna().sum()}")
print(f"gaps:    {(s.index.to_series().diff().dt.days > 1).sum()}")

Date
2023-01-01    1343.77
2023-01-02    1678.78
2023-01-03    1569.62
Name: Revenue, dtype: float64

span:    2023-01-01 to 2024-12-30
days:    730
missing: 0
gaps:    0


### Block 2  (`c2.py`)

In [4]:
by_dow = s.groupby(s.index.day_name()).mean()
order = ["Monday", "Tuesday", "Wednesday", "Thursday",
         "Friday", "Saturday", "Sunday"]
overall = s.mean()

for d in order:
    print(f"  {d:<10} {by_dow[d]:>8,.0f}   {by_dow[d]/overall:>5.2f}x")
print(f"  {'overall':<10} {overall:>8,.0f}")

  Monday        2,441    0.95x
  Tuesday       2,338    0.91x
  Wednesday     2,349    0.91x
  Thursday      2,525    0.98x
  Friday        2,936    1.14x
  Saturday      3,303    1.28x
  Sunday        2,127    0.83x
  overall       2,573


### Block 3  (`c3.py`)

In [5]:
roll = s.rolling(7, center=True).mean()      # 7 days kills the weekly cycle

print(f"raw    standard deviation: {s.std():>8,.0f}")
print(f"smooth standard deviation: {roll.std():>8,.0f}")
print(f"share of variation the smoothing removes: "
      f"{1 - roll.var()/s.var():.1%}")

raw    standard deviation:      636
smooth standard deviation:      461
share of variation the smoothing removes: 47.3%


### Block 4  (`c4.py`)

In [6]:
import numpy as np

H = 28                                   # horizon: four weeks
train, test = s[:-H], s[-H:]
mae = lambda pred: np.mean(np.abs(test.values - np.asarray(pred)))

print(f"train ends {train.index.max().date()},  test is {len(test)} days")
for name, pred in [
        ("last value",       np.repeat(train.iloc[-1], H)),
        ("last 28-day mean", np.repeat(train.iloc[-28:].mean(), H)),
        ("seasonal naive",   train.iloc[-7:].values.tolist() * 4)]:
    print(f"  {name:<18} MAE {mae(pred):>7,.0f}")

train ends 2024-12-02,  test is 28 days
  last value         MAE     957
  last 28-day mean   MAE     852
  seasonal naive     MAE     758


### Block 5  (`c5.py`)

In [7]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

fit = ExponentialSmoothing(train, trend="add",
                           seasonal="add", seasonal_periods=7).fit()
hw = fit.forecast(H)

sn = train.iloc[-7:].values.tolist() * 4
print(f"  seasonal naive     MAE {mae(sn):>7,.0f}")
print(f"  Holt-Winters       MAE {mae(hw):>7,.0f}")
for k in ("smoothing_level", "smoothing_trend", "smoothing_seasonal"):
    print(f"  {k:<20} {fit.params[k]:.3f}")

  seasonal naive     MAE     758
  Holt-Winters       MAE     728
  smoothing_level      0.210
  smoothing_trend      0.000
  smoothing_seasonal   0.083


<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


### Block 6  (`c6.py`)

In [8]:
from sklearn.ensemble import RandomForestRegressor

f = pd.DataFrame({"y": s})
f["dow"], f["month"] = f.index.dayofweek, f.index.month
f["lag_7"]   = f["y"].shift(7)            # same weekday, one week back
f["lag_14"]  = f["y"].shift(14)
f["roll_28"] = f["y"].shift(7).rolling(28).mean()
f = f.dropna()

X = ["dow", "month", "lag_7", "lag_14", "roll_28"]
tr, te = f[:-H], f[-H:]
m = RandomForestRegressor(n_estimators=300,
                          random_state=0).fit(tr[X], tr["y"])

err = np.abs(te["y"] - m.predict(te[X]))
print(f"  random forest      MAE {err.mean():>7,.0f}")
for n, v in sorted(zip(X, m.feature_importances_), key=lambda p: -p[1]):
    print(f"  {n:<10} {v:.3f}")

  random forest      MAE     795
  lag_7      0.802
  lag_14     0.115
  month      0.035
  roll_28    0.034
  dow        0.015


### Block 7  (`c7.py`)

In [9]:
def fold(cut):
    tr, te = s[:cut], s[cut:cut + H]
    rep = np.array(tr.iloc[-7:].values.tolist() * 4)
    sn = np.mean(np.abs(te.values - rep))
    hw = ExponentialSmoothing(tr, trend="add", seasonal="add",
                              seasonal_periods=7).fit().forecast(H)
    return {"seasonal naive": sn,
            "Holt-Winters": np.mean(np.abs(te.values - hw.values)),
            "fold ends": s.index[cut + H - 1].date()}

cuts = [len(s) - H * k for k in range(6, 0, -1)]
t = pd.DataFrame([fold(c) for c in cuts]).set_index("fold ends")
print(t.round(0).to_string())
print("\nmean across six folds")
print(t.mean().round(0).to_string())

<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
<site-packages>/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


            seasonal naive  Holt-Winters
fold ends                               
2024-08-12           167.0         179.0
2024-09-09           252.0         181.0
2024-10-07           261.0         249.0
2024-11-04           251.0         275.0
2024-12-02           154.0         112.0
2024-12-30           758.0         728.0

mean across six folds
seasonal naive    307.0
Holt-Winters      287.0
